<a href="https://colab.research.google.com/github/sakshumvij/GNN_Stroke_Outcome_Prediction/blob/main/GNN_Stroke_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **GNN Stroke Outcome Prediction Code**

Paper Title: Analyzing Structural Brain Connectivity with Graph Neural Networks to Predict Intracerebral Hemorrhage Recovery


This notebook implements and evaluates a graph neural network (GNN) framework for predicting functional outcome following hemorrhagic stroke using structural brain connectivity. Four models are trained and compared: a multimodal GNN combining structural connectivity with clinical features, a graph-only GNN, a LASSO connectivity model, and a clinical baseline logistic regression. A post-hoc edge importance explainer identifies the connectivity patterns most predictive of poor outcome.

# Dependencies and Set Up

In [ ]:
# Install Dependencies

!pip install "tensorflow-gnn>=1.0.0" -q
!pip install nilearn -q

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

This notebook requires **TensorFlow 2.x** with the TF-GNN extension for graph neural network operations. The **TF_USE_LEGACY_KERAS** flag must be set before import to ensure compatibility between TF-GNN and the Keras API. Run this cell first to confirm correct library versions are installed before executing any model cells.

In [ ]:
import tensorflow as tf
import tensorflow_gnn as tfgnn

print("TensorFlow version:", tf.__version__)
print("TF-GNN version:", tfgnn.__version__)

# Multimodal GNN
Weighted GraphSAGE network that fuses structural connectivity graphs with clinical features via a gated joint embedding, trained with focal loss to handle class imbalance.

In [ ]:
# Multimodal Brain Connectivity GNN + Clinical Features

# Imports
import os
import gc
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_gnn as tfgnn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    confusion_matrix, roc_curve, classification_report,
)

# Set random seeds and file paths.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CSV_PATH = "/content/ROSE_connectivity_binary_mRS.csv"
MAT_DIR = "/content/filtered_patients/ROSE_Connectivity_Matrices"
CLINICAL_CSV = "/content/clinical_and_demographics.csv"

WEIGHTS_PATH = "/content/multimodal_gnn.weights.h5"
PREPROCESSOR_PATH = "/content/multimodal_clinical_preprocessing.joblib"
TEST_IDS_PATH = "/content/gnn_test_subject_ids.csv"
PLOT_PATH = "/content/multimodal_gnn_results.png"

# Define model settings and dataset split proportions.
HIDDEN_DIM, NUM_LAYERS, NODE_FEAT_DIM = 48, 3, 5
DROPOUT_RATE, L2_REG, BATCH_SIZE = 0.45, 8e-4, 16
EPOCHS, PATIENCE, BOOTSTRAP_N = 400, 60, 1000
LR_INIT, CLINICAL_DIM = 5e-4, 2

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.60, 0.15, 0.25
EDGE_PERCENTILE = 80  # Preserved from the supplied multimodal model.
CLINICAL_FEATURES = ["ICH_Volume", "ICH_Location_2"]


# Define the loss function and early stopping behavior.
def focal_loss(gamma=2.0, alpha=0.25, label_smoothing=0.05):
    bce = tf.keras.losses.BinaryCrossentropy(
        label_smoothing=label_smoothing,
        reduction=tf.keras.losses.Reduction.NONE,
    )

    def loss_fn(y_true, y_pred):
        y_true = tf.reshape(y_true, [-1])
        y_pred = tf.reshape(y_pred, [-1])
        bce_val = bce(y_true[:, None], y_pred[:, None])
        p_t = tf.where(y_true == 1.0, y_pred, 1.0 - y_pred)
        return tf.reduce_mean(
            alpha * tf.pow(1.0 - p_t, gamma) * bce_val
        )

    return loss_fn


class SmoothedEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, monitor="val_auc", patience=PATIENCE, window=5):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.window = window
        self.history = []
        self.wait = 0
        self.best = -np.inf
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        value = (logs or {}).get(self.monitor)
        if value is None:
            return

        self.history.append(value)
        smoothed = np.mean(self.history[-self.window:])

        # Save the model weights when the smoothed score improves.
        if smoothed > self.best:
            self.best = smoothed
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                if self.best_weights is not None:
                    self.model.set_weights(self.best_weights)

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


# Calculate node features from each brain connectivity matrix.
def fast_node_features(connectivity):
    W = np.abs(connectivity).astype(np.float64)
    n = W.shape[0]

    A = (W > 0).astype(np.float64)
    degree = A.sum(axis=1)
    strength = W.sum(axis=1)

    W_cbrt = np.cbrt(W)
    triangles = (W_cbrt @ W_cbrt * W_cbrt).diagonal()
    denom = degree * (degree - 1)
    clustering = np.divide(
        triangles, denom,
        out=np.zeros_like(triangles),
        where=denom > 0,
    )

    col_sums = W.sum(axis=0)
    col_sums[col_sums == 0] = 1.0
    T = W / col_sums[None, :]

    pr = np.ones(n, dtype=np.float64) / n
    for _ in range(10):
        pr = 0.85 * (T @ pr) + 0.15 / n

    # Split positive edge weights around their median.
    positive = W[W > 0]
    median = np.median(positive) if positive.size else 0.0
    mod_a = ((W > median) & (W > 0)).sum(axis=1)
    mod_b = ((W <= median) & (W > 0)).sum(axis=1)
    degree_safe = np.where(degree > 0, degree, 1.0)

    participation = (
        1.0 - (mod_a / degree_safe) ** 2
        - (mod_b / degree_safe) ** 2
    )

    # Standardize the five node features.
    features = np.stack(
        [degree, strength, clustering, pr, participation],
        axis=1,
    )
    std = features.std(axis=0)
    std[std < 1e-8] = 1.0

    return ((features - features.mean(axis=0)) / std).astype(np.float32)


# Randomly remove some edges during training augmentation.
def augment_connectivity(connectivity, edge_dropout=0.10):
    W = connectivity.copy()
    rows, cols = np.nonzero(np.triu(W, k=1))
    n_drop = int(edge_dropout * len(rows))

    if n_drop:
        for idx in np.random.choice(len(rows), n_drop, replace=False):
            r, c = rows[idx], cols[idx]
            W[r, c] = W[c, r] = 0.0

    return W


# Convert a connectivity matrix into a graph.
def graph_from_mat(path, augment=False):
    connectivity = loadmat(path)["connectivity"].astype(np.float32)

    if augment:
        connectivity = augment_connectivity(connectivity)

    # Keep edges at or above the selected positive-weight percentile.
    positive = connectivity[connectivity > 0]
    threshold = (
        np.percentile(positive, EDGE_PERCENTILE)
        if positive.size else 0.0
    )

    src, tgt = np.where(
        (connectivity >= threshold) & (connectivity > 0)
    )
    weights = connectivity[src, tgt]

    if len(weights) > 1:
        std = weights.std()
        weights = (weights - weights.mean()) / (std if std > 0 else 1.0)

    node_features = fast_node_features(connectivity)
    n_nodes = connectivity.shape[0]

    return tfgnn.GraphTensor.from_pieces(
        node_sets={
            "brain_region": tfgnn.NodeSet.from_fields(
                sizes=tf.constant([n_nodes]),
                features={"hidden_state": tf.constant(node_features)},
            )
        },
        edge_sets={
            "connectivity": tfgnn.EdgeSet.from_fields(
                sizes=tf.constant([len(src)]),
                adjacency=tfgnn.Adjacency.from_indices(
                    source=(
                        "brain_region",
                        tf.constant(src.astype(np.int32)),
                    ),
                    target=(
                        "brain_region",
                        tf.constant(tgt.astype(np.int32)),
                    ),
                ),
                features={
                    "weight": tf.constant(weights.astype(np.float32))
                },
            )
        },
        context=tfgnn.Context.from_fields(
            features={}, sizes=tf.constant([1])
        ),
    )


# Define the weighted graph convolution layer.
class WeightedSAGEConv(tf.keras.layers.Layer):
    def __init__(self, out_dim, l2_reg=L2_REG, **kwargs):
        super().__init__(**kwargs)
        reg = tf.keras.regularizers.l2(l2_reg)
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(
                out_dim, use_bias=False, kernel_regularizer=reg
            ),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
        ])

    def call(self, graph, training=False):
        if graph.rank > 0:
            graph = graph.merge_batch_to_components()

        node_h = graph.node_sets["brain_region"]["hidden_state"]
        edge = graph.edge_sets["connectivity"]
        edge_w = edge.features["weight"][:, None]
        src, tgt = edge.adjacency.source, edge.adjacency.target
        n = tf.shape(node_h)[0]

        # Aggregate neighboring node features using edge weights.
        agg = tf.math.unsorted_segment_sum(
            tf.gather(node_h, src) * edge_w, tgt, n
        )
        denom = tf.math.unsorted_segment_sum(
            tf.abs(edge_w), tgt, n
        ) + 1e-6

        new_h = self.mlp(
            tf.concat([node_h, agg / denom], axis=-1),
            training=training,
        )
        return graph.replace_features(
            node_sets={"brain_region": {"hidden_state": new_h}}
        )


# Define the GNN that combines graph and clinical features.
class GNNWithClinical(tf.keras.Model):
    def __init__(self):
        super().__init__()
        reg = tf.keras.regularizers.l2(L2_REG)

        self.input_proj = tf.keras.Sequential([
            tf.keras.layers.Dense(
                HIDDEN_DIM, use_bias=False, kernel_regularizer=reg
            ),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
        ])
        self.convs = [
            WeightedSAGEConv(HIDDEN_DIM)
            for _ in range(NUM_LAYERS)
        ]
        self.residuals = [
            tf.keras.layers.Dense(
                HIDDEN_DIM, use_bias=False, kernel_regularizer=reg
            )
            for _ in range(NUM_LAYERS)
        ]
        self.pool = tfgnn.keras.layers.Pool(
            reduce_type="mean",
            tag=tfgnn.CONTEXT,
            node_set_name="brain_region",
            feature_name="hidden_state",
        )
        self.clinical_proj = tf.keras.Sequential([
            tf.keras.layers.Dense(8, kernel_regularizer=reg),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
            tf.keras.layers.Dense(4, kernel_regularizer=reg),
        ])

        joint_dim = HIDDEN_DIM + 4
        self.gate = tf.keras.layers.Dense(
            joint_dim, activation="sigmoid", kernel_regularizer=reg
        )
        self.value = tf.keras.layers.Dense(
            joint_dim, activation="swish", kernel_regularizer=reg
        )
        self.dropout = tf.keras.layers.Dropout(DROPOUT_RATE)
        self.output_layer = tf.keras.layers.Dense(
            1, activation="sigmoid", kernel_regularizer=reg
        )

    def call(self, inputs, training=False):
        graph, clinical = inputs

        if graph.rank > 0:
            graph = graph.merge_batch_to_components()

        h = self.input_proj(
            graph.node_sets["brain_region"]["hidden_state"],
            training=training,
        )
        graph = graph.replace_features(
            node_sets={"brain_region": {"hidden_state": h}}
        )

        # Apply graph layers and add residual connections.
        for conv, residual in zip(self.convs, self.residuals):
            prev = graph.node_sets["brain_region"]["hidden_state"]
            graph = conv(graph, training=training)
            h = graph.node_sets["brain_region"]["hidden_state"] + residual(prev)
            graph = graph.replace_features(
                node_sets={"brain_region": {"hidden_state": h}}
            )

        # Combine pooled graph features with clinical features.
        graph_emb = self.pool(graph)
        clinical_emb = self.clinical_proj(clinical, training=training)
        joint = tf.concat([graph_emb, clinical_emb], axis=-1)
        gated = self.dropout(
            self.gate(joint) * self.value(joint), training=training
        )
        return self.output_layer(gated)


# Define the graph input format and dataset helpers.
def make_graph_spec(n_nodes):
    return tfgnn.GraphTensorSpec.from_piece_specs(
        node_sets_spec={
            "brain_region": tfgnn.NodeSetSpec.from_field_specs(
                features_spec={
                    "hidden_state": tf.TensorSpec(
                        (n_nodes, NODE_FEAT_DIM), tf.float32
                    )
                },
                sizes_spec=tf.TensorSpec((1,), tf.int32),
            )
        },
        edge_sets_spec={
            "connectivity": tfgnn.EdgeSetSpec.from_field_specs(
                features_spec={
                    "weight": tf.TensorSpec((None,), tf.float32)
                },
                sizes_spec=tf.TensorSpec((1,), tf.int32),
                adjacency_spec=tfgnn.AdjacencySpec.from_incident_node_sets(
                    "brain_region",
                    "brain_region",
                    tf.TensorSpec((None,), tf.int32),
                ),
            )
        },
        context_spec=tfgnn.ContextSpec.from_field_specs(
            features_spec={},
            sizes_spec=tf.TensorSpec((1,), tf.int32),
        ),
    )


def make_samples(subjects, augment=False):
    samples = [
        (graph_from_mat(s["mat_path"]), s["clinical_scaled"], s["label"])
        for s in subjects
    ]
    if augment:
        samples += [
            (
                graph_from_mat(s["mat_path"], augment=True),
                s["clinical_scaled"],
                s["label"],
            )
            for s in subjects
        ]
    return samples


def make_dataset(samples, graph_spec, shuffle=False):
    def gen():
        for graph, clinical, label in samples:
            yield (
                (graph, np.asarray(clinical, dtype=np.float32)),
                np.float32(label),
            )

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            (graph_spec, tf.TensorSpec((CLINICAL_DIM,), tf.float32)),
            tf.TensorSpec((), tf.float32),
        ),
    )
    if shuffle:
        ds = ds.shuffle(len(samples), reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Calculate test metrics and bootstrap confidence intervals.
def metric_values(y, prob, threshold):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    return {
        "AUROC": roc_auc_score(y, prob),
        "Accuracy": accuracy_score(y, pred),
        "F1": f1_score(y, pred, zero_division=0),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
    }


def bootstrap_cis(y, prob, threshold, n_boot=BOOTSTRAP_N):
    rng = np.random.default_rng(SEED)
    groups = [np.flatnonzero(y == cls) for cls in (0, 1)]
    boot = {key: [] for key in metric_values(y, prob, threshold)}

    # Resample each outcome class separately.
    for _ in range(n_boot):
        idx = np.concatenate([
            rng.choice(group, len(group), replace=True)
            for group in groups
        ])
        for key, value in metric_values(
            y[idx], prob[idx], threshold
        ).items():
            boot[key].append(value)

    point = metric_values(y, prob, threshold)
    cis = {
        key: np.percentile(values, [2.5, 97.5])
        for key, values in boot.items()
    }
    return point, cis


# Select the threshold that maximizes sensitivity plus specificity minus one.
def select_threshold(y_val, p_val):
    fpr, tpr, thresholds = roc_curve(y_val, p_val)
    valid = np.isfinite(thresholds)
    if not valid.any():
        return 0.5
    return float(thresholds[valid][np.argmax((tpr - fpr)[valid])])


# Load outcome labels and clinical data.
labels_df = pd.read_csv(CSV_PATH)
label_dict = dict(zip(
    labels_df["Subject"].astype(int),
    labels_df["mRS_binary"].astype(int),
))

clinical_df = pd.read_csv(CLINICAL_CSV)
clinical_df["ID_int"] = pd.to_numeric(
    clinical_df["ID"].astype(str).str.replace("-", "", regex=False),
    errors="coerce",
)
clinical_df = clinical_df.dropna(subset=["ID_int"]).copy()
clinical_df["ID_int"] = clinical_df["ID_int"].astype(int)

if clinical_df["ID_int"].duplicated().any():
    raise ValueError("Duplicate clinical subject IDs found.")

clinical_df["ICH_Volume"] = pd.to_numeric(
    clinical_df["ICH_Volume"], errors="coerce"
).replace([np.inf, -np.inf], np.nan)

location = pd.to_numeric(
    clinical_df["ICH_Location"], errors="coerce"
).replace([np.inf, -np.inf], np.nan)

# Create the binary clinical feature for location category 2.
clinical_df["ICH_Location_2"] = np.where(
    location.notna(), (location == 2).astype(float), np.nan
)

clinical_lookup = {
    int(row.ID_int): row[CLINICAL_FEATURES].to_numpy(dtype=np.float32)
    for _, row in clinical_df.iterrows()
}

# Match matrix files to subjects with known outcome labels.
subjects_pool = []
for filename in sorted(os.listdir(MAT_DIR)):
    if not filename.endswith(".mat"):
        continue
    try:
        sid = int(filename.split("_")[0].replace("-", ""))
    except ValueError:
        continue

    if sid in label_dict:
        subjects_pool.append({
            "id": sid,
            "mat_path": os.path.join(MAT_DIR, filename),
            "label": float(label_dict[sid]),
            "clinical_raw": clinical_lookup.get(
                sid, np.full(CLINICAL_DIM, np.nan, dtype=np.float32)
            ),
        })

if not subjects_pool:
    raise ValueError("No labeled subjects with matching .mat files found.")
if len({s["id"] for s in subjects_pool}) != len(subjects_pool):
    raise ValueError("Multiple .mat files found for a subject ID.")


# Split subjects into training, validation, and test sets.
all_subjects = np.array(subjects_pool, dtype=object)
all_labels = np.array([s["label"] for s in subjects_pool])

trainval_subj, test_subj, trainval_y, _ = train_test_split(
    all_subjects,
    all_labels,
    test_size=TEST_FRAC,
    stratify=all_labels,
    random_state=SEED,
)

val_relative = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
train_subj, val_subj, train_y, _ = train_test_split(
    trainval_subj,
    trainval_y,
    test_size=val_relative,
    stratify=trainval_y,
    random_state=SEED,
)

# Fit imputation and scaling on training subjects ONLY.
def raw_clinical(subjects):
    return np.stack([s["clinical_raw"] for s in subjects]).astype(np.float32)

train_raw, val_raw, test_raw = map(
    raw_clinical, (train_subj, val_subj, test_subj)
)

all_missing = np.isnan(train_raw).all(axis=0)
if all_missing.any():
    missing = np.array(CLINICAL_FEATURES)[all_missing].tolist()
    raise ValueError(f"Clinical features entirely missing in training: {missing}")

clinical_imputer = SimpleImputer(strategy="median")
train_clin = clinical_imputer.fit_transform(train_raw)
val_clin = clinical_imputer.transform(val_raw)
test_clin = clinical_imputer.transform(test_raw)

clinical_scaler = StandardScaler()
train_clin = clinical_scaler.fit_transform(train_clin).astype(np.float32)
val_clin = clinical_scaler.transform(val_clin).astype(np.float32)
test_clin = clinical_scaler.transform(test_clin).astype(np.float32)

# Attach the processed clinical features to each subject.
for subjects, values in [
    (train_subj, train_clin),
    (val_subj, val_clin),
    (test_subj, test_clin),
]:
    for subject, value in zip(subjects, values):
        subject["clinical_scaled"] = value

# Save the held-out test subject IDs.
pd.DataFrame({
    "ID_int": [int(s["id"]) for s in test_subj]
}).to_csv(TEST_IDS_PATH, index=False)

print(
    f"Split: train={len(train_subj)}, val={len(val_subj)}, "
    f"test={len(test_subj)}"
)
print(
    "Subjects with no clinical row: "
    f"{sum(s['id'] not in clinical_lookup for s in subjects_pool)}"
)


# Build datasets and train the multimodal GNN.
num_nodes = loadmat(subjects_pool[0]["mat_path"])["connectivity"].shape[0]
graph_spec = make_graph_spec(num_nodes)

train_samples = make_samples(list(train_subj), augment=True)
val_samples = make_samples(list(val_subj))
test_samples = make_samples(list(test_subj))

train_ds = make_dataset(train_samples, graph_spec, shuffle=True)
val_ds = make_dataset(val_samples, graph_spec)
test_ds = make_dataset(test_samples, graph_spec)

n_pos = float(train_y.sum())
alpha = (len(train_y) - n_pos) / (len(train_y) + 1e-8)
steps_per_epoch = max(1, len(train_samples) // BATCH_SIZE)

lr = tf.keras.optimizers.schedules.CosineDecayRestarts(
    LR_INIT,
    50 * steps_per_epoch,
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-5,
)

model = GNNWithClinical()
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr),
    loss=focal_loss(alpha=alpha),
    metrics=[tf.keras.metrics.AUC(name="auc")],
)

ckpt = "/tmp/best_multimodal_gnn.weights.h5"
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1,
    callbacks=[
        SmoothedEarlyStopping(),
        tf.keras.callbacks.ModelCheckpoint(
            ckpt,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),
    ],
)
model.load_weights(ckpt)
if os.path.exists(ckpt):
    os.remove(ckpt)


# Select the threshold using validation predictions and evaluate the test set.
y_val = np.array([float(s["label"]) for s in val_subj])
p_val = model.predict(val_ds, verbose=0).ravel()
threshold = select_threshold(y_val, p_val)

y_test = np.array([float(s["label"]) for s in test_subj])
p_test = model.predict(test_ds, verbose=0).ravel()

point, cis = bootstrap_cis(y_test, p_test, threshold)

print(f"\nValidation-selected threshold: {threshold:.4f}")
print("Multimodal GNN held-out test results (95% bootstrap CI):")
for name, value in point.items():
    lo, hi = cis[name]
    print(f"{name:<14} {value:.4f} ({lo:.4f}, {hi:.4f})")

print("\n" + classification_report(
    y_test,
    (p_test >= threshold).astype(int),
    target_names=["Favorable (mRS 0-2)", "Unfavorable (mRS 3-6)"],
    zero_division=0,
))


# Save the ROC plot, model weights, and clinical preprocessing objects.
fpr, tpr, _ = roc_curve(y_test, p_test)
fig, ax = plt.subplots(figsize=(6, 5))
lo, hi = cis["AUROC"]
ax.plot(
    fpr, tpr,
    label=f"AUC={point['AUROC']:.3f} (95% CI {lo:.3f}-{hi:.3f})",
)
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title="Multimodal GNN — held-out test set",
)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=200)
plt.close(fig)

model.save_weights(WEIGHTS_PATH)
joblib.dump({
    "imputer": clinical_imputer,
    "scaler": clinical_scaler,
    "features": CLINICAL_FEATURES,
    "threshold": threshold,
}, PREPROCESSOR_PATH)

print(f"\nSaved weights: {WEIGHTS_PATH}")
print(f"Saved clinical preprocessing and threshold: {PREPROCESSOR_PATH}")
print(f"Saved held-out test IDs: {TEST_IDS_PATH}")
print(f"Saved ROC plot: {PLOT_PATH}")

gc.collect()
tf.keras.backend.clear_session()

# GNN Only Model
Model identical in architecture to the multimodal GNN but trained on structural connectivity alone, allowing direct analysis of the clinical feature contribution.

In [ ]:
# Pure Brain Connectivity GNN — no clinical features

# Imports
import os
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_gnn as tfgnn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    confusion_matrix, roc_curve, classification_report,
)

# Set random seeds and file paths.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CSV_PATH = "/content/ROSE_connectivity_binary_mRS.csv"
MAT_DIR = "/content/filtered_patients/ROSE_Connectivity_Matrices"

WEIGHTS_PATH = "/content/pure_gnn.weights.h5"
TEST_IDS_PATH = "/content/gnn_test_subject_ids.csv"
PLOT_PATH = "/content/pure_gnn_results.png"

# Define model settings and dataset split proportions.
HIDDEN_DIM, NUM_LAYERS, NODE_FEAT_DIM = 48, 3, 5
DROPOUT_RATE, L2_REG, BATCH_SIZE = 0.45, 8e-4, 16
EPOCHS, PATIENCE, BOOTSTRAP_N = 400, 60, 1000
LR_INIT = 5e-4

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.60, 0.15, 0.25
EDGE_PERCENTILE = 80  # Preserved from the supplied pure-GNN model.


# Define the loss function and early stopping behavior.
def focal_loss(gamma=2.0, alpha=0.25, label_smoothing=0.05):
    bce = tf.keras.losses.BinaryCrossentropy(
        label_smoothing=label_smoothing,
        reduction=tf.keras.losses.Reduction.NONE,
    )

    def loss_fn(y_true, y_pred):
        y_true = tf.reshape(y_true, [-1])
        y_pred = tf.reshape(y_pred, [-1])
        bce_val = bce(y_true[:, None], y_pred[:, None])
        p_t = tf.where(y_true == 1.0, y_pred, 1.0 - y_pred)
        return tf.reduce_mean(
            alpha * tf.pow(1.0 - p_t, gamma) * bce_val
        )

    return loss_fn


class SmoothedEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, monitor="val_auc", patience=PATIENCE, window=5):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.window = window
        self.history = []
        self.wait = 0
        self.best = -np.inf
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        value = (logs or {}).get(self.monitor)
        if value is None:
            return

        self.history.append(value)
        smoothed = np.mean(self.history[-self.window:])

        # Save the model weights when the smoothed score improves.
        if smoothed > self.best:
            self.best = smoothed
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                if self.best_weights is not None:
                    self.model.set_weights(self.best_weights)

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


# Calculate node features from each brain connectivity matrix.
def fast_node_features(connectivity):
    W = np.abs(connectivity).astype(np.float64)
    n = W.shape[0]

    A = (W > 0).astype(np.float64)
    degree = A.sum(axis=1)
    strength = W.sum(axis=1)

    W_cbrt = np.cbrt(W)
    triangles = (W_cbrt @ W_cbrt * W_cbrt).diagonal()
    denom = degree * (degree - 1)
    clustering = np.divide(
        triangles, denom,
        out=np.zeros_like(triangles),
        where=denom > 0,
    )

    col_sums = W.sum(axis=0)
    col_sums[col_sums == 0] = 1.0
    T = W / col_sums[None, :]

    pr = np.ones(n, dtype=np.float64) / n
    for _ in range(10):
        pr = 0.85 * (T @ pr) + 0.15 / n

    # Split positive edge weights around their median.
    positive = W[W > 0]
    median = np.median(positive) if positive.size else 0.0
    mod_a = ((W > median) & (W > 0)).sum(axis=1)
    mod_b = ((W <= median) & (W > 0)).sum(axis=1)
    degree_safe = np.where(degree > 0, degree, 1.0)

    participation = (
        1.0 - (mod_a / degree_safe) ** 2
        - (mod_b / degree_safe) ** 2
    )

    # Standardize the five node features.
    features = np.stack(
        [degree, strength, clustering, pr, participation],
        axis=1,
    )
    std = features.std(axis=0)
    std[std < 1e-8] = 1.0

    return ((features - features.mean(axis=0)) / std).astype(np.float32)


# Randomly remove some edges during training augmentation.
def augment_connectivity(connectivity, edge_dropout=0.10):
    W = connectivity.copy()
    rows, cols = np.nonzero(np.triu(W, k=1))
    n_drop = int(edge_dropout * len(rows))

    if n_drop:
        for idx in np.random.choice(len(rows), n_drop, replace=False):
            r, c = rows[idx], cols[idx]
            W[r, c] = W[c, r] = 0.0

    return W


# Convert a connectivity matrix into a graph.
def graph_from_mat(path, augment=False):
    connectivity = loadmat(path)["connectivity"].astype(np.float32)

    if augment:
        connectivity = augment_connectivity(connectivity)

    # Keep edges at or above the selected positive-weight percentile.
    positive = connectivity[connectivity > 0]
    threshold = (
        np.percentile(positive, EDGE_PERCENTILE)
        if positive.size else 0.0
    )

    src, tgt = np.where(
        (connectivity >= threshold) & (connectivity > 0)
    )
    weights = connectivity[src, tgt]

    if len(weights) > 1:
        std = weights.std()
        weights = (weights - weights.mean()) / (std if std > 0 else 1.0)

    node_features = fast_node_features(connectivity)
    n_nodes = connectivity.shape[0]

    return tfgnn.GraphTensor.from_pieces(
        node_sets={
            "brain_region": tfgnn.NodeSet.from_fields(
                sizes=tf.constant([n_nodes]),
                features={"hidden_state": tf.constant(node_features)},
            )
        },
        edge_sets={
            "connectivity": tfgnn.EdgeSet.from_fields(
                sizes=tf.constant([len(src)]),
                adjacency=tfgnn.Adjacency.from_indices(
                    source=(
                        "brain_region",
                        tf.constant(src.astype(np.int32)),
                    ),
                    target=(
                        "brain_region",
                        tf.constant(tgt.astype(np.int32)),
                    ),
                ),
                features={
                    "weight": tf.constant(weights.astype(np.float32))
                },
            )
        },
        context=tfgnn.Context.from_fields(
            features={}, sizes=tf.constant([1])
        ),
    )


# Define the weighted graph convolution layer.
class WeightedSAGEConv(tf.keras.layers.Layer):
    def __init__(self, out_dim, l2_reg=L2_REG, **kwargs):
        super().__init__(**kwargs)
        reg = tf.keras.regularizers.l2(l2_reg)
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(
                out_dim, use_bias=False, kernel_regularizer=reg
            ),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
        ])

    def call(self, graph, training=False):
        if graph.rank > 0:
            graph = graph.merge_batch_to_components()

        node_h = graph.node_sets["brain_region"]["hidden_state"]
        edge = graph.edge_sets["connectivity"]
        edge_w = edge.features["weight"][:, None]
        src, tgt = edge.adjacency.source, edge.adjacency.target
        n = tf.shape(node_h)[0]

        # Aggregate neighboring node features using edge weights.
        agg = tf.math.unsorted_segment_sum(
            tf.gather(node_h, src) * edge_w, tgt, n
        )
        denom = tf.math.unsorted_segment_sum(
            tf.abs(edge_w), tgt, n
        ) + 1e-6

        new_h = self.mlp(
            tf.concat([node_h, agg / denom], axis=-1),
            training=training,
        )
        return graph.replace_features(
            node_sets={"brain_region": {"hidden_state": new_h}}
        )


# Define the GNN that uses only brain connectivity features.
class PureGNN(tf.keras.Model):
    def __init__(self):
        super().__init__()
        reg = tf.keras.regularizers.l2(L2_REG)

        self.input_proj = tf.keras.Sequential([
            tf.keras.layers.Dense(
                HIDDEN_DIM, use_bias=False, kernel_regularizer=reg
            ),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
        ])
        self.convs = [
            WeightedSAGEConv(HIDDEN_DIM)
            for _ in range(NUM_LAYERS)
        ]
        self.residuals = [
            tf.keras.layers.Dense(
                HIDDEN_DIM, use_bias=False, kernel_regularizer=reg
            )
            for _ in range(NUM_LAYERS)
        ]
        self.pool = tfgnn.keras.layers.Pool(
            reduce_type="mean",
            tag=tfgnn.CONTEXT,
            node_set_name="brain_region",
            feature_name="hidden_state",
        )
        self.gate = tf.keras.layers.Dense(
            HIDDEN_DIM, activation="sigmoid", kernel_regularizer=reg
        )
        self.value = tf.keras.layers.Dense(
            HIDDEN_DIM, activation="swish", kernel_regularizer=reg
        )
        self.dropout = tf.keras.layers.Dropout(DROPOUT_RATE)
        self.output_layer = tf.keras.layers.Dense(
            1, activation="sigmoid", kernel_regularizer=reg
        )

    def call(self, graph, training=False):
        if graph.rank > 0:
            graph = graph.merge_batch_to_components()

        h = self.input_proj(
            graph.node_sets["brain_region"]["hidden_state"],
            training=training,
        )
        graph = graph.replace_features(
            node_sets={"brain_region": {"hidden_state": h}}
        )

        # Apply graph layers and add residual connections.
        for conv, residual in zip(self.convs, self.residuals):
            prev = graph.node_sets["brain_region"]["hidden_state"]
            graph = conv(graph, training=training)
            h = graph.node_sets["brain_region"]["hidden_state"] + residual(prev)
            graph = graph.replace_features(
                node_sets={"brain_region": {"hidden_state": h}}
            )

        # Pool node features and predict the outcome.
        emb = self.pool(graph)
        gated = self.dropout(
            self.gate(emb) * self.value(emb), training=training
        )
        return self.output_layer(gated)


# Define the graph input format and dataset helpers.
def make_graph_spec(n_nodes):
    return tfgnn.GraphTensorSpec.from_piece_specs(
        node_sets_spec={
            "brain_region": tfgnn.NodeSetSpec.from_field_specs(
                features_spec={
                    "hidden_state": tf.TensorSpec(
                        (n_nodes, NODE_FEAT_DIM), tf.float32
                    )
                },
                sizes_spec=tf.TensorSpec((1,), tf.int32),
            )
        },
        edge_sets_spec={
            "connectivity": tfgnn.EdgeSetSpec.from_field_specs(
                features_spec={
                    "weight": tf.TensorSpec((None,), tf.float32)
                },
                sizes_spec=tf.TensorSpec((1,), tf.int32),
                adjacency_spec=tfgnn.AdjacencySpec.from_incident_node_sets(
                    "brain_region",
                    "brain_region",
                    tf.TensorSpec((None,), tf.int32),
                ),
            )
        },
        context_spec=tfgnn.ContextSpec.from_field_specs(
            features_spec={},
            sizes_spec=tf.TensorSpec((1,), tf.int32),
        ),
    )


def make_samples(subjects, augment=False):
    samples = [
        (graph_from_mat(s["mat_path"]), s["label"])
        for s in subjects
    ]
    if augment:
        samples += [
            (graph_from_mat(s["mat_path"], augment=True), s["label"])
            for s in subjects
        ]
    return samples


def make_dataset(samples, graph_spec, shuffle=False):
    def gen():
        for graph, label in samples:
            yield graph, np.float32(label)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            graph_spec,
            tf.TensorSpec((), tf.float32),
        ),
    )
    if shuffle:
        ds = ds.shuffle(len(samples), reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Calculate test metrics and bootstrap confidence intervals.
def metric_values(y, prob, threshold):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    return {
        "AUROC": roc_auc_score(y, prob),
        "Accuracy": accuracy_score(y, pred),
        "F1": f1_score(y, pred, zero_division=0),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
    }


def bootstrap_cis(y, prob, threshold, n_boot=BOOTSTRAP_N):
    rng = np.random.default_rng(SEED)
    groups = [np.flatnonzero(y == cls) for cls in (0, 1)]
    boot = {key: [] for key in metric_values(y, prob, threshold)}

    # Resample each outcome class separately.
    for _ in range(n_boot):
        idx = np.concatenate([
            rng.choice(group, len(group), replace=True)
            for group in groups
        ])
        for key, value in metric_values(
            y[idx], prob[idx], threshold
        ).items():
            boot[key].append(value)

    point = metric_values(y, prob, threshold)
    cis = {
        key: np.percentile(values, [2.5, 97.5])
        for key, values in boot.items()
    }
    return point, cis


# Select the threshold that maximizes sensitivity plus specificity minus one.
def select_threshold(y_val, p_val):
    fpr, tpr, thresholds = roc_curve(y_val, p_val)
    valid = np.isfinite(thresholds)
    if not valid.any():
        return 0.5
    return float(thresholds[valid][np.argmax((tpr - fpr)[valid])])


# Load outcome labels and find matching connectivity matrices.
labels_df = pd.read_csv(CSV_PATH)
label_dict = dict(zip(
    labels_df["Subject"].astype(int),
    labels_df["mRS_binary"].astype(int),
))

subjects_pool = []
for filename in sorted(os.listdir(MAT_DIR)):
    if not filename.endswith(".mat"):
        continue
    try:
        sid = int(filename.split("_")[0].replace("-", ""))
    except ValueError:
        continue

    if sid in label_dict:
        subjects_pool.append({
            "id": sid,
            "mat_path": os.path.join(MAT_DIR, filename),
            "label": float(label_dict[sid]),
        })

if not subjects_pool:
    raise ValueError("No labeled subjects with matching .mat files found.")
if len({s["id"] for s in subjects_pool}) != len(subjects_pool):
    raise ValueError("Multiple .mat files found for a subject ID.")

# Split subjects into training, validation, and test sets.
all_subjects = np.array(subjects_pool, dtype=object)
all_labels = np.array([s["label"] for s in subjects_pool])

trainval_subj, test_subj, trainval_y, _ = train_test_split(
    all_subjects,
    all_labels,
    test_size=TEST_FRAC,
    stratify=all_labels,
    random_state=SEED,
)

val_relative = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
train_subj, val_subj, train_y, _ = train_test_split(
    trainval_subj,
    trainval_y,
    test_size=val_relative,
    stratify=trainval_y,
    random_state=SEED,
)

# Save the held-out test subject IDs.
pd.DataFrame({
    "ID_int": [int(s["id"]) for s in test_subj]
}).to_csv(TEST_IDS_PATH, index=False)

print(
    f"Split: train={len(train_subj)}, val={len(val_subj)}, "
    f"test={len(test_subj)}"
)


# Build datasets and train the pure GNN.
num_nodes = loadmat(subjects_pool[0]["mat_path"])["connectivity"].shape[0]
graph_spec = make_graph_spec(num_nodes)

train_samples = make_samples(list(train_subj), augment=True)
val_samples = make_samples(list(val_subj))
test_samples = make_samples(list(test_subj))

train_ds = make_dataset(train_samples, graph_spec, shuffle=True)
val_ds = make_dataset(val_samples, graph_spec)
test_ds = make_dataset(test_samples, graph_spec)

n_pos = float(train_y.sum())
alpha = (len(train_y) - n_pos) / (len(train_y) + 1e-8)
steps_per_epoch = max(1, len(train_samples) // BATCH_SIZE)

lr = tf.keras.optimizers.schedules.CosineDecayRestarts(
    LR_INIT,
    50 * steps_per_epoch,
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-5,
)

model = PureGNN()
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr),
    loss=focal_loss(alpha=alpha),
    metrics=[tf.keras.metrics.AUC(name="auc")],
)

ckpt = "/tmp/best_pure_gnn.weights.h5"
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1,
    callbacks=[
        SmoothedEarlyStopping(),
        tf.keras.callbacks.ModelCheckpoint(
            ckpt,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),
    ],
)
model.load_weights(ckpt)
if os.path.exists(ckpt):
    os.remove(ckpt)


# Select the threshold using validation predictions and evaluate the test set.
y_val = np.array([float(s["label"]) for s in val_subj])
p_val = model.predict(val_ds, verbose=0).ravel()
threshold = select_threshold(y_val, p_val)

y_test = np.array([float(s["label"]) for s in test_subj])
p_test = model.predict(test_ds, verbose=0).ravel()

point, cis = bootstrap_cis(y_test, p_test, threshold)

print(f"\nValidation-selected threshold: {threshold:.4f}")
print("Pure GNN held-out test results (95% bootstrap CI):")
for name, value in point.items():
    lo, hi = cis[name]
    print(f"{name:<14} {value:.4f} ({lo:.4f}, {hi:.4f})")

print("\n" + classification_report(
    y_test,
    (p_test >= threshold).astype(int),
    target_names=["Favorable (mRS 0-2)", "Unfavorable (mRS 3-6)"],
    zero_division=0,
))


# Save the ROC plot and model weights.
fpr, tpr, _ = roc_curve(y_test, p_test)
fig, ax = plt.subplots(figsize=(6, 5))
lo, hi = cis["AUROC"]
ax.plot(
    fpr, tpr,
    label=f"AUC={point['AUROC']:.3f} (95% CI {lo:.3f}-{hi:.3f})",
)
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title="Pure GNN — held-out test set",
)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=200)
plt.close(fig)

model.save_weights(WEIGHTS_PATH)
print(f"\nSaved weights: {WEIGHTS_PATH}")
print(f"Saved held-out test IDs: {TEST_IDS_PATH}")
print(f"Saved ROC plot: {PLOT_PATH}")

gc.collect()
tf.keras.backend.clear_session()

# LASSO Connectivity Model
L1-regularised logistic regression applied to connectivity matrices, using nested cross-validation to identify predictive structural edges without graph operations.

In [ ]:
# LASSO Connectivity Model

# Imports
import os
import glob
import warnings
import pickle
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)

warnings.filterwarnings("ignore")

# Set file paths and model settings.
CSV_PATH = "/content/ROSE_connectivity_binary_mRS.csv"
MAT_DIR = "/content/filtered_patients/ROSE_Connectivity_Matrices"
RESULTS_DIR = "./results"

SUBJECT_COL = "Subject"
LABEL_COL = "mRS_binary"
MAT_VAR_NAMES = ["connectivity", "matrix", "CM", "sc", "FC", "data"]

C_VALUES = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]

OUTER_CV_FOLDS = 5
INNER_CV_FOLDS = 5
UPPER_TRI_ONLY = True
TOP_N = 20
RANDOM_STATE = 42
TEST_FRAC = 0.25


# Normalize subject IDs for matching.
def normalise_subject_id(raw_id):
    return str(raw_id).replace("-", "").replace(" ", "").strip()


# Reproduce the held-out test-subject split.
def get_shared_test_subject_ids(csv_path, mat_dir):
    df = pd.read_csv(csv_path)
    label_dict = dict(zip(df[SUBJECT_COL].astype(int), df[LABEL_COL]))

    candidate_ids = []
    candidate_labels = []

    for filename in sorted(os.listdir(mat_dir)):
        if not filename.endswith(".mat"):
            continue

        raw_id = os.path.basename(filename).split("_")[0].replace("-", "")
        try:
            sub_id = int(raw_id)
        except ValueError:
            continue

        if sub_id not in label_dict:
            continue

        candidate_ids.append(sub_id)
        candidate_labels.append(float(label_dict[sub_id]))

    _, test_ids, _, _ = train_test_split(
        np.array(candidate_ids),
        np.array(candidate_labels),
        test_size=TEST_FRAC,
        stratify=candidate_labels,
        random_state=RANDOM_STATE,
    )

    return {normalise_subject_id(sid) for sid in test_ids}


# Load a structural connectivity matrix from a .mat file.
def load_mat_matrix(filepath):
    try:
        mat = sio.loadmat(filepath)
    except Exception as e:
        raise IOError(f"Cannot read {filepath}: {e}")

    for key in MAT_VAR_NAMES:
        if key in mat and isinstance(mat[key], np.ndarray):
            arr = mat[key]
            if arr.ndim == 2:
                return arr.astype(float)

    for key, val in mat.items():
        if key.startswith("_"):
            continue
        if isinstance(val, np.ndarray) and val.ndim == 2:
            return val.astype(float)

    raise ValueError(f"No 2-D matrix found in {filepath}. Keys: {list(mat.keys())}")


# Convert connectivity matrices into feature vectors.
def matrix_to_vector(matrix, upper_tri_only=True):
    if upper_tri_only:
        idx = np.triu_indices(matrix.shape[0], k=1)
        return matrix[idx]
    return matrix.flatten()


# Create labels for individual connectivity edges.
def build_edge_labels(n_nodes, upper_tri_only=True):
    if upper_tri_only:
        rows, cols = np.triu_indices(n_nodes, k=1)
        return [f"edge_{r}_{c}" for r, c in zip(rows, cols)]
    return [f"node_{i}" for i in range(n_nodes * n_nodes)]


# Load outcome labels and connectivity matrices.
def load_dataset(csv_path, mat_dir):
    print("=" * 60)
    print("Loading dataset …")

    df = pd.read_csv(csv_path)
    if df.columns[0].startswith("Unnamed"):
        df = df.iloc[:, 1:]

    print(f"  CSV rows: {len(df)}  |  columns: {list(df.columns)}")

    mat_files = glob.glob(os.path.join(mat_dir, "*.mat"))
    print(f"  .mat files found: {len(mat_files)}")

    mat_lookup = {}
    for fp in mat_files:
        raw_id = Path(fp).name.split("_")[0]
        norm_id = normalise_subject_id(raw_id)
        mat_lookup[norm_id] = fp

    X_list, y_list, matched_ids = [], [], []
    missing = []

    for _, row in df.iterrows():
        norm_id = normalise_subject_id(row[SUBJECT_COL])
        if norm_id not in mat_lookup:
            missing.append(row[SUBJECT_COL])
            continue
        try:
            mat = load_mat_matrix(mat_lookup[norm_id])
        except Exception as e:
            print(f"  [WARN] Skipping {row[SUBJECT_COL]}: {e}")
            continue
        X_list.append(matrix_to_vector(mat, UPPER_TRI_ONLY))
        y_list.append(int(row[LABEL_COL]))
        matched_ids.append(row[SUBJECT_COL])

    if missing:
        print(f"  [WARN] {len(missing)} subject(s) with no .mat file: {missing[:5]}…")

    X = np.array(X_list)
    y = np.array(y_list)

    print(f"  Matched subjects : {len(matched_ids)}")
    print(f"  Feature length   : {X.shape[1]}")
    print(f"  Class dist       → 0 (good): {(y==0).sum()}  |  1 (poor): {(y==1).sum()}")

    n_nodes = int((1 + np.sqrt(1 + 8 * X.shape[1])) / 2) if UPPER_TRI_ONLY else int(np.sqrt(X.shape[1]))
    feature_names = build_edge_labels(n_nodes, UPPER_TRI_ONLY)
    return X, y, matched_ids, feature_names


# Remove invalid values and identify features with non-zero variance.
def clean_features(X):
    X = X.copy()
    X[~np.isfinite(X)] = np.nan
    col_medians = np.nanmedian(X, axis=0)
    inds = np.where(np.isnan(X))
    X[inds] = np.take(col_medians, inds[1])
    non_zero_cols = np.where(X.std(axis=0) > 1e-10)[0]
    print(f"  Columns with non-zero variance: {len(non_zero_cols)} / {X.shape[1]}")
    return X[:, non_zero_cols], non_zero_cols


# Impute missing values and filter features using training data only.
def clean_features_train_test(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    X_train[~np.isfinite(X_train)] = np.nan
    X_test[~np.isfinite(X_test)] = np.nan

    col_medians = np.nanmedian(X_train, axis=0)
    col_medians[~np.isfinite(col_medians)] = 0.0

    train_inds = np.where(np.isnan(X_train))
    X_train[train_inds] = np.take(col_medians, train_inds[1])

    test_inds = np.where(np.isnan(X_test))
    X_test[test_inds] = np.take(col_medians, test_inds[1])

    non_zero_cols = np.where(X_train.std(axis=0) > 1e-10)[0]
    print(f"  Columns with non-zero variance: {len(non_zero_cols)} / {X_train.shape[1]}")

    return X_train[:, non_zero_cols], X_test[:, non_zero_cols], non_zero_cols


# Build the LASSO logistic-regression pipeline.
def build_pipeline():
    clf = LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])


# Run nested cross-validation to select the regularization strength.
def nested_cv(X, y, feature_names):
    print("\n" + "=" * 60)
    print("Running nested cross-validation …")

    outer_cv = StratifiedKFold(
        n_splits=OUTER_CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    inner_cv = StratifiedKFold(
        n_splits=INNER_CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    param_grid = {"clf__C": C_VALUES}

    y_pred_all = np.zeros(len(y), dtype=int)
    y_proba_all = np.zeros(len(y))
    cv_log = []
    coef_folds = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        gs = GridSearchCV(
            estimator=build_pipeline(),
            param_grid=param_grid,
            cv=inner_cv,
            scoring="roc_auc",
            n_jobs=-1,
            refit=True,
        )
        gs.fit(X_tr, y_tr)

        best = gs.best_params_
        print(f"  Fold {fold_idx+1}: best params = {best}  |  inner AUC = {gs.best_score_:.3f}")

        y_pred_all[test_idx] = gs.predict(X_te)
        y_proba_all[test_idx] = gs.predict_proba(X_te)[:, 1]
        coef_folds.append(gs.best_estimator_.named_steps["clf"].coef_[0])

        cv_log.append({
            "fold": fold_idx + 1,
            "best_C": best["clf__C"],
            "inner_auc": gs.best_score_,
            "test_auc": roc_auc_score(y_te, y_proba_all[test_idx]),
        })

    return y_pred_all, y_proba_all, coef_folds, cv_log


# Fit the final LASSO model using the selected regularization strength.
def fit_final_model(X, y, cv_log, feature_names):
    best_C = np.median([r["best_C"] for r in cv_log])
    print(f"\n  Final LASSO model: C={best_C}")

    pipe = build_pipeline()
    pipe.named_steps["clf"].C = best_C
    pipe.fit(X, y)

    coef = pipe.named_steps["clf"].coef_[0]
    nonzero = np.where(coef != 0)[0]
    feat_df = pd.DataFrame({
        "feature": [feature_names[i] for i in nonzero],
        "coefficient": coef[nonzero],
    }).sort_values("coefficient", key=abs, ascending=False)

    print(f"  Non-zero features selected: {len(feat_df)} / {len(feature_names)}")
    return pipe, feat_df


# Save and reload the fitted LASSO model.
def save_model(pipeline, auc, out_dir, kept_cols):

    filename = f"lasso_auc{auc:.4f}_seed{RANDOM_STATE}.pkl"
    filepath = os.path.join(out_dir, filename)

    payload = {
        "pipeline": pipeline,
        "auc": auc,
        "kept_cols": kept_cols,
        "model_type": "lasso",
        "random_state": RANDOM_STATE,
        "upper_tri": UPPER_TRI_ONLY,
    }
    with open(filepath, "wb") as f:
        pickle.dump(payload, f)

    print(f"  Model saved → {filepath}")
    return filepath


def load_saved_model(pkl_path):
    with open(pkl_path, "rb") as f:
        return pickle.load(f)


# Generate the held-out test-set ROC curve.
def save_roc_curve(y_true, y_proba, out_dir):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve – Held-Out Test Set")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=150)
    plt.close(fig)
    print(f"  Held-out test AUC: {auc:.3f}")
    return auc


# Generate the held-out test-set confusion matrix.
def save_confusion_matrix(y_true, y_pred, out_dir):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Good (0-2)", "Poor (3-6)"],
                yticklabels=["Good (0-2)", "Poor (3-6)"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (Held-Out Test Set)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
    plt.close(fig)


# Plot the highest-magnitude selected connectivity edges.
def save_top_features(feat_df, out_dir, top_n=TOP_N):
    top = feat_df.head(top_n)
    colors = ["tomato" if c < 0 else "steelblue" for c in top["coefficient"]]
    fig, ax = plt.subplots(figsize=(8, max(4, top_n * 0.35)))
    ax.barh(top["feature"][::-1], top["coefficient"][::-1], color=colors[::-1])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Coefficient")
    ax.set_title(f"Top {top_n} Selected Connectivity Edges")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "top_features.png"), dpi=150)
    plt.close(fig)


# Plot coefficient stability across cross-validation folds.
def save_fold_coef_stability(coef_folds, feature_names, out_dir, top_n=TOP_N):
    coef_mat = np.array(coef_folds)
    mean_coef = coef_mat.mean(axis=0)
    std_coef = coef_mat.std(axis=0)

    nonzero_counts = (coef_mat != 0).sum(axis=0)
    candidate_idx = np.where(nonzero_counts > 1)[0]
    if len(candidate_idx) == 0:
        candidate_idx = np.argsort(np.abs(mean_coef))[::-1]

    top_idx = candidate_idx[np.argsort(np.abs(mean_coef[candidate_idx]))[::-1][:top_n]]

    fig, ax = plt.subplots(figsize=(8, max(4, len(top_idx) * 0.35)))
    y_pos = np.arange(len(top_idx))
    ax.barh(y_pos, mean_coef[top_idx[::-1]], xerr=std_coef[top_idx[::-1]],
            color="steelblue", alpha=0.8, ecolor="grey", capsize=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([feature_names[i] for i in top_idx[::-1]])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Mean Coefficient ± SD across folds")
    ax.set_title("Feature Stability across CV Folds")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "feature_stability.png"), dpi=150)
    plt.close(fig)


# Calculate threshold-dependent classification metrics.
def _point_metrics(y, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
    return {
        "Test AUROC": roc_auc_score(y, y_proba),
        "Test Accuracy (%)": 100 * (tp + tn) / (tp + tn + fp + fn),
        "F1-Score": f1_score(y, y_pred, zero_division=0),
        "Sensitivity (%)": 100 * tp / (tp + fn) if (tp + fn) else np.nan,
        "Specificity (%)": 100 * tn / (tn + fp) if (tn + fp) else np.nan,
    }


# Estimate 95% confidence intervals using bootstrap resampling.
def bootstrap_ci(y, y_pred, y_proba, n_boot=2000, alpha=0.05):
    rng = np.random.default_rng(RANDOM_STATE)
    point = _point_metrics(y, y_pred, y_proba)
    boots = {k: [] for k in point}
    n = len(y)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:      # AUROC undefined
            continue
        for k, v in _point_metrics(y[idx], y_pred[idx], y_proba[idx]).items():
            boots[k].append(v)
    out = {}
    for k in point:
        lo, hi = np.nanpercentile(boots[k], [100 * alpha / 2, 100 * (1 - alpha / 2)])
        out[k] = (point[k], lo, hi)
    return out


# Estimate the confidence interval for validation AUROC.
def val_auc_ci(inner_aucs, alpha=0.05):
    a = np.asarray(inner_aucs, dtype=float)
    m = a.mean()
    h = stats.t.ppf(1 - alpha / 2, len(a) - 1) * a.std(ddof=1) / np.sqrt(len(a))
    return m, m - h, m + h


# Save validation and held-out test metrics.
def save_summary_table(y, y_pred, y_proba, cv_df, out_dir):
    res = {"Val AUROC": val_auc_ci(cv_df["inner_auc"])}
    res.update(bootstrap_ci(y, y_pred, y_proba))
    order = ["Val AUROC", "Test AUROC", "Test Accuracy (%)",
             "F1-Score", "Sensitivity (%)", "Specificity (%)"]
    rows = []
    for k in order:
        v, lo, hi = res[k]
        d = 1 if "%" in k else 3
        rows.append({"Metric": k, "Value": round(v, d),
                     "CI_low": round(lo, d), "CI_high": round(hi, d),
                     "Formatted": f"{v:.{d}f} ({lo:.{d}f}-{hi:.{d}f})"})
    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(out_dir, "summary_metrics.csv"), index=False)
    print("\nSummary (95% CI):")
    print(summary[["Metric", "Formatted"]].to_string(index=False))
    return summary


# Run the full LASSO analysis.
def main():
    os.makedirs(RESULTS_DIR, exist_ok=True)

    X_raw, y, subject_ids, feature_names_all = load_dataset(CSV_PATH, MAT_DIR)

    test_id_keys = get_shared_test_subject_ids(CSV_PATH, MAT_DIR)
    test_mask = np.array([
        normalise_subject_id(sid) in test_id_keys for sid in subject_ids
    ])

    if test_mask.sum() != len(test_id_keys):
        raise ValueError(
            f"Expected to match all {len(test_id_keys)} shared test subjects, "
            f"but matched {int(test_mask.sum())}."
        )

    X_train_raw, X_test_raw = X_raw[~test_mask], X_raw[test_mask]
    y_train, y_test = y[~test_mask], y[test_mask]

    print(f"\nShared held-out test set: {len(y_test)} subjects")
    print(f"Training set: {len(y_train)} subjects")

    print("\nCleaning features …")
    X_train, X_test, kept_cols = clean_features_train_test(X_train_raw, X_test_raw)
    feature_names = [feature_names_all[i] for i in kept_cols]

    train_cv_pred, train_cv_proba, coef_folds, cv_log = nested_cv(
        X_train, y_train, feature_names
    )

    print("\n" + "=" * 60)
    print("Classification Report (held-out test set):")

    cv_df = pd.DataFrame(cv_log)
    cv_df.to_csv(os.path.join(RESULTS_DIR, "cv_results.csv"), index=False)
    print(f"\nCV fold summary:\n{cv_df.to_string(index=False)}")

    final_pipe, feat_df = fit_final_model(X_train, y_train, cv_log, feature_names)
    y_pred = final_pipe.predict(X_test)
    y_proba = final_pipe.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=["Good outcome (mRS 0-2)", "Poor outcome (mRS 3-6)"]
    )
    print(report)
    with open(os.path.join(RESULTS_DIR, "classification_report.txt"), "w") as f:
        f.write(report)

    save_summary_table(y_test, y_pred, y_proba, cv_df, RESULTS_DIR)

    print("\nSaving plots …")
    auc = save_roc_curve(y_test, y_proba, RESULTS_DIR)
    save_confusion_matrix(y_test, y_pred, RESULTS_DIR)
    save_fold_coef_stability(coef_folds, feature_names, RESULTS_DIR)

    feat_df.to_csv(os.path.join(RESULTS_DIR, "selected_features.csv"), index=False)
    save_top_features(feat_df, RESULTS_DIR)

    train_oof_auc = roc_auc_score(y_train, train_cv_proba)
    save_model(final_pipe, train_oof_auc, RESULTS_DIR, kept_cols)

    print("\n" + "=" * 60)
    print("Done. Results written to:", RESULTS_DIR)
    print(f"  Held-out test AUC       : {auc:.3f}")
    print(f"  Non-zero edges (final)  : {len(feat_df)}")
    print("=" * 60)


if __name__ == "__main__":
    main()

# Clinical Baseline
Logistic regression trained on demographic and clinical features to establish a prediction benchmark.

In [ ]:
# Clinical Baseline Without Gender

# Imports
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, confusion_matrix
)

# Set file paths and model settings.
LABELS_CSV = "/content/ROSE_connectivity_binary_mRS.csv"
CLINICAL_CSV = "/content/clinical_and_demographics.csv"
MAT_DIR = "/content/filtered_patients/ROSE_Connectivity_Matrices"
MODEL_PATH = "/content/clinical_baseline_without_gender.joblib"

SUBJECT_COL = "Subject"
LABEL_COL = "mRS_binary"
CLINICAL_ID_COL = "ID"

num_features = ["Age", "ICH_Volume"]
cat_features = ["Race", "ICH_Location", "ICH_Laterality"]
feature_cols = num_features + cat_features

RANDOM_STATE = 42
TEST_FRAC = 0.25
BOOTSTRAP_RESAMPLES = 1000
BOOTSTRAP_RANDOM_STATE = 42


# Load outcome labels and clinicl data.
labels_raw = pd.read_csv(LABELS_CSV)
clinical_raw = pd.read_csv(CLINICAL_CSV)

labels_raw.columns = labels_raw.columns.str.strip()
clinical_raw.columns = clinical_raw.columns.str.strip()

# Check that all required columns are present.
for columns, required, name in [
    (labels_raw.columns, [SUBJECT_COL, LABEL_COL], "outcome"),
    (clinical_raw.columns, [CLINICAL_ID_COL] + feature_cols, "clinical"),
]:
    missing = set(required) - set(columns)
    if missing:
        raise ValueError(f"Missing {name} columns: {sorted(missing)}")

# Convert subject IDs and outcome labels to numeric values.
labels_raw[SUBJECT_COL] = pd.to_numeric(
    labels_raw[SUBJECT_COL], errors="coerce"
)
labels_raw[LABEL_COL] = pd.to_numeric(
    labels_raw[LABEL_COL], errors="coerce"
)
clinical_raw[CLINICAL_ID_COL] = pd.to_numeric(
    clinical_raw[CLINICAL_ID_COL], errors="coerce"
)

labels_raw = labels_raw.dropna(subset=[SUBJECT_COL, LABEL_COL]).copy()
clinical_raw = clinical_raw.dropna(subset=[CLINICAL_ID_COL]).copy()

labels_raw[SUBJECT_COL] = labels_raw[SUBJECT_COL].astype(int)
clinical_raw[CLINICAL_ID_COL] = clinical_raw[CLINICAL_ID_COL].astype(int)

if not labels_raw[LABEL_COL].isin([0, 1]).all():
    raise ValueError(f"{LABEL_COL} must contain only 0 and 1.")

labels_raw[LABEL_COL] = labels_raw[LABEL_COL].astype(int)

# Check for duplicate subject IDs.
for data, col in [
    (labels_raw, SUBJECT_COL),
    (clinical_raw, CLINICAL_ID_COL),
]:
    if data[col].duplicated().any():
        raise ValueError(f"Duplicate subject IDs in {col}.")


# Reproduce the exact LASSO subject split.
label_dict = dict(zip(labels_raw[SUBJECT_COL], labels_raw[LABEL_COL]))
candidate_ids, candidate_labels = [], []

for filename in sorted(os.listdir(MAT_DIR)):
    if not filename.endswith(".mat"):
        continue

    try:
        subject_id = int(filename.split("_")[0].replace("-", ""))
    except ValueError:
        continue

    if subject_id in label_dict:
        candidate_ids.append(subject_id)
        candidate_labels.append(float(label_dict[subject_id]))

if not candidate_ids:
    raise ValueError("No matching labeled subjects and .mat files were found.")

if len(candidate_ids) != len(set(candidate_ids)):
    raise ValueError("Multiple .mat files were found for the same subject ID.")

# Preserve the original file order and stratified split procedure.
train_ids, test_ids, _, _ = train_test_split(
    np.array(candidate_ids),
    np.array(candidate_labels),
    test_size=TEST_FRAC,
    stratify=candidate_labels,
    random_state=RANDOM_STATE,
)

eligible_ids = set(candidate_ids)
train_ids, test_ids = set(train_ids), set(test_ids)


# Merge clinical features with outcome labels and prepare the split.
labels = labels_raw[[SUBJECT_COL, LABEL_COL]].rename(
    columns={SUBJECT_COL: "ID", LABEL_COL: "mRS_binary"}
)
clinical = clinical_raw[[CLINICAL_ID_COL] + feature_cols].rename(
    columns={CLINICAL_ID_COL: "ID"}
)

df = labels[labels["ID"].isin(eligible_ids)].merge(
    clinical, on="ID", how="left", validate="one_to_one"
)

# Treat unparseable numeric values and infinities as missing.
for col in num_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df[num_features] = df[num_features].replace([np.inf, -np.inf], np.nan)

# Treat blank categorical values as missing.
df[cat_features] = df[cat_features].replace(r"^\s*$", np.nan, regex=True)

train_df = df[df["ID"].isin(train_ids)].sort_values("ID")
test_df = df[df["ID"].isin(test_ids)].sort_values("ID")

if len(train_df) != len(train_ids) or len(test_df) != len(test_ids):
    raise ValueError("Merged clinical data do not match the shared subject split.")

X_train, y_train = train_df[feature_cols], train_df["mRS_binary"].to_numpy()
X_test, y_test = test_df[feature_cols], test_df["mRS_binary"].to_numpy()

if X_train.isna().all().any():
    missing = X_train.columns[X_train.isna().all()].tolist()
    raise ValueError(f"Entirely missing training features: {missing}")

if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
    raise ValueError("Both outcome classes must be present in train and test sets.")


# Define preprocessing and train the clinical model.
# Preprocessing is fitted on training data only.
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

model.fit(X_train, y_train)


# Calculate held-out test metrics.
probabilities = model.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype(int)

def calculate_metrics(y_true, y_prob, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "AUROC": roc_auc_score(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
    }

point_estimates = calculate_metrics(y_test, probabilities, predictions)


# Calculate stratified bootstrap 95% confidence intervals.
rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)
class_indices = [
    np.flatnonzero(y_test == cls) for cls in [0, 1]
]

bootstrap_results = {metric: [] for metric in point_estimates}

for _ in range(BOOTSTRAP_RESAMPLES):
    # Resample each outcome class with replacement, preserving class counts.
    indices = np.concatenate([
        rng.choice(indices, size=len(indices), replace=True)
        for indices in class_indices
    ])

    metrics = calculate_metrics(
        y_test[indices],
        probabilities[indices],
        predictions[indices],
    )

    for metric, value in metrics.items():
        bootstrap_results[metric].append(value)

confidence_intervals = {
    metric: np.percentile(values, [2.5, 97.5])
    for metric, values in bootstrap_results.items()
}

print("\n" + "=" * 65)
print("CLINICAL BASELINE — HELD-OUT TEST RESULTS")
print("=" * 65)
print(f"Training subjects: {len(y_train)}")
print(f"Test subjects:     {len(y_test)}")
print(f"Bootstrap resamples: {BOOTSTRAP_RESAMPLES}")
print("\nMetric             Estimate     95% CI")
print("-" * 52)

for metric, estimate in point_estimates.items():
    lower, upper = confidence_intervals[metric]
    print(f"{metric:<19} {estimate:.4f}       ({lower:.4f}, {upper:.4f})")


# Save the fitted clinical model pipeline.
joblib.dump(model, MODEL_PATH)
print(f"\nSaved model to: {MODEL_PATH}")

# Edge Importance Explainer
Post-hoc soft edge-mask explainer applied across all validation subjects to identify and visualise the structural connections most influential to the model's outcome predictions.

In [ ]:
# GNN Edge Importance Explainer

# Imports
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_gnn as tfgnn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

from nilearn import plotting as nl_plotting
from IPython.display import display

# Path Configurations
ATLAS_CSV_PATH = "/content/HCP_SUIT_FreeSurferSubcortical_coords (1) (1).csv"

# Output Paths
GLASS_BRAIN_PATH = "/content/gnn_edge_importance_2d.png"
REGION_BAR_PATH = "/content/gnn_region_importance_bar.png"
IMPORTANCE_STD_PATH = "/content/gnn_importance_std.png"
REGION_RANKINGS_PATH = "/content/region_rankings.csv"


# Atlas Setup
# Load region names and MNI coordinates from the HCP/SUIT/FreeSurfer atlas CSV.

atlas_df = pd.read_csv(ATLAS_CSV_PATH)

# Sort by ROI ID so row index matches connectivity matrix ordering
atlas_df = atlas_df.sort_values("ROI ID").reset_index(drop=True)

region_names = list(atlas_df["ROI Name"])
coords = atlas_df[["x", "y", "z"]].values  # (NUM_NODES, 3) MNI coordinates

print(f"Loaded {len(region_names)} regions")
print(f"Coords shape: {coords.shape}")
print(f"\nFirst 5 regions:")
for i in range(5):
    print(f"[{i}] {region_names[i]:20s}  MNI: {coords[i]}")

# Confirm atlas size matches the GNN node count
assert len(region_names) == NUM_NODES, (
    f"Atlas has {len(region_names)} regions but model expects {NUM_NODES}. "
    f"Check your CSV"
)
print(f"\nAtlas matches model: {NUM_NODES} regions")


# Edge Mask Explainer
# Learns a soft per-edge mask in [0,1] where high values indicate
# edges whose removal most changes the model's prediction.

class EdgeMaskExplainer(tf.keras.Model):
    def __init__(self, frozen_model, graph, clinical_vec):
        super().__init__()
        self.frozen_model = frozen_model
        self.clinical_vec = clinical_vec
        num_edges = graph.edge_sets["connectivity"].sizes[0].numpy()
        self.edge_logits = tf.Variable(
            tf.zeros([num_edges], dtype=tf.float32),
            trainable=True,
            name="edge_logits",
        )

    def call(self, graph, training=False):
        mask = tf.nn.sigmoid(self.edge_logits)
        masked_graph = graph.replace_features(
            edge_sets={
                "connectivity": {
                    "weight": graph.edge_sets["connectivity"]["weight"] * mask
                }
            }
        )
        return self.frozen_model((masked_graph, self.clinical_vec), training=False)


# Single-Subject Explanation
# Optimizes edge masks for one subject to approximate which edges
# drive the prediction, with a sparsity penalty to avoid trivial solutions.

def explain_single(frozen_model, graph, clinical_vec,
                   steps=400, lr=0.03, sparsity_coef=0.02):
    frozen_model.trainable = False

    clin = tf.convert_to_tensor(clinical_vec, dtype=tf.float32)
    if clin.ndim == 1:
        clin = tf.expand_dims(clin, 0)

    original_pred = frozen_model((graph, clin), training=False)
    explainer = EdgeMaskExplainer(frozen_model, graph, clin)
    opt = tf.keras.optimizers.Adam(lr)

    for _ in range(steps):
        with tf.GradientTape() as tape:
            pred = explainer(graph, training=False)
            pred_loss = tf.keras.losses.binary_crossentropy(original_pred, pred)
            sparsity_loss = tf.reduce_mean(tf.nn.sigmoid(explainer.edge_logits))
            loss = pred_loss + sparsity_coef * sparsity_loss
        grads = tape.gradient(loss, [explainer.edge_logits])
        opt.apply_gradients(zip(grads, [explainer.edge_logits]))

    importance = tf.nn.sigmoid(explainer.edge_logits).numpy()
    src = graph.edge_sets["connectivity"].adjacency.source.numpy()
    tgt = graph.edge_sets["connectivity"].adjacency.target.numpy()
    return importance, src, tgt


# Population-Level Explanation
# Runs explain_single per subject, then aligns all masks onto a shared
# union edge index. Subjects missing a given edge receive importance = 0.

def explain_population(frozen_model, dataset, subject_list, scaler, max_subjects=None, steps=400, lr=0.03, sparsity_coef=0.02):
    subjs = subject_list
    if max_subjects is not None:
        subjs = subject_list[:max_subjects]

    per_subject = []
    n_total = len(subjs)

    unbatched = dataset.unbatch()
    for i, ((graph, clin_t), label_t) in enumerate(unbatched):
        if max_subjects is not None and i >= max_subjects:
            break
        print(f"  Subject {i+1}/{n_total}  label={int(label_t.numpy())} ...",
              end="\r", flush=True)
        importance, src, tgt = explain_single(
            frozen_model, graph, clin_t,
            steps=steps, lr=lr, sparsity_coef=sparsity_coef,
        )
        edge_w_subj = graph.edge_sets["connectivity"]["weight"].numpy()
        per_subject.append((src, tgt, importance, edge_w_subj))

    print(f"\nExplained {len(per_subject)} subjects.")

    # Build union edge index using canonical (min, max) node pairs
    union_set = {}
    for src, tgt, _, _ in per_subject:
        for u, v in zip(src, tgt):
            key = (min(int(u), int(v)), max(int(u), int(v)))
            if key not in union_set:
                union_set[key] = len(union_set)

    U = len(union_set)
    union_src = np.array([k[0] for k in union_set.keys()], dtype=np.int32)
    union_tgt = np.array([k[1] for k in union_set.keys()], dtype=np.int32)

    print(f"Union edge index size: {U}  "
          f"(individual subjects had "
          f"{min(len(s[0]) for s in per_subject)}–"
          f"{max(len(s[0]) for s in per_subject)} edges)")

    # Map each subject's mask onto the union index
    all_masks = np.zeros((len(per_subject), U), dtype=np.float32)
    weight_sum = np.zeros(U, dtype=np.float64)
    weight_cnt = np.zeros(U, dtype=np.float64)

    for i, (src, tgt, importance, edge_w_subj) in enumerate(per_subject):
        for u, v, imp, w in zip(src, tgt, importance, edge_w_subj):
            key = (min(int(u), int(v)), max(int(u), int(v)))
            j = union_set[key]
            all_masks[i, j] = imp
            weight_sum[j]  += w
            weight_cnt[j]  += 1.0

    mean_mask = all_masks.mean(axis=0)
    edge_w_mean = np.where(weight_cnt > 0, weight_sum / weight_cnt, 0.0)

    return mean_mask, union_src, union_tgt, edge_w_mean, all_masks


# Ranked Edge and Region Tables
# Aggregates edge importance scores to the region level by summing
# contributions from all edges incident to each region.

def build_importance_tables(mean_mask, edge_src, edge_tgt, edge_w, region_names):
    ranked = sorted(
        zip(edge_src, edge_tgt, edge_w, mean_mask),
        key=lambda x: x[3], reverse=True,
    )
    edge_df = pd.DataFrame(ranked,columns=["source", "target", "weight", "importance"])
    edge_df["source_region"] = edge_df["source"].map(lambda i: region_names[i])
    edge_df["target_region"] = edge_df["target"].map(lambda i: region_names[i])

    num_regions = len(region_names)
    region_scores = np.zeros(num_regions, dtype=np.float64)
    for u, v, w, m in ranked:
        region_scores[int(u)] += m
        region_scores[int(v)] += m

    mx = region_scores.max()
    if mx > 0:
        region_scores /= mx

    region_df = pd.DataFrame({
        "region":     region_names,
        "importance": region_scores,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    return edge_df, region_df


# Connectivity Matrix
# Builds a symmetric importance matrix from the top-ranked edges.

def build_conn_matrix(edge_df, num_regions, num_top=100):
    conn = np.zeros((num_regions, num_regions), dtype=np.float64)
    for _, row in edge_df.head(num_top).iterrows():
        u, v, imp = int(row["source"]), int(row["target"]), row["importance"]
        conn[u, v] = imp
        conn[v, u] = imp
    return conn


# Visualizations

def plot_3d_connectome_inline(conn_matrix, coords,
                               title="GNN Edge Importance",
                               edge_thresh="85%", node_size=6, linewidth=3):
    # Interactive 3-D connectome rendered inline in Colab.
    view = nl_plotting.view_connectome(
        conn_matrix, coords,
        edge_threshold=edge_thresh,
        colorbar=True,
        node_size=node_size,
        linewidth=linewidth,
        title=title,
    )
    display(view)


def plot_2d_glass_brain(conn_matrix, coords, region_scores, save_path=GLASS_BRAIN_PATH, edge_thresh="90%"):
    # Three-view glass brain; node size scaled by per-region importance.
    node_sizes = 20 + 80 * region_scores

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, mode, ttl in zip(axes,
                              ["x","y","z"],
                              ["Sagittal", "Coronal", "Axial"]):
        nl_plotting.plot_connectome(
            conn_matrix, coords,
            edge_threshold=edge_thresh,
            node_size=node_sizes,
            display_mode=mode,
            colorbar=False,
            axes=ax,
            title=ttl,
        )
    plt.suptitle("GNN Edge Importance — Glass Brain", fontsize=13, y=1.02)
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"2-D glass brain → {save_path}")


def plot_region_bar(region_df, top_n=20,
                    save_path=REGION_BAR_PATH):
    # Horizontal bar chart of the top-N regions by aggregated edge importance.
    top = region_df.head(top_n).iloc[::-1]
    cmap = cm.get_cmap("YlOrRd")
    norm = mcolors.Normalize(vmin=top["importance"].min(),
                              vmax=top["importance"].max())
    colors = [cmap(norm(v)) for v in top["importance"]]

    fig, ax = plt.subplots(figsize=(9, 0.45 * top_n + 1.5))
    bars = ax.barh(top["region"], top["importance"],
                   color=colors, edgecolor="white", height=0.7)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlabel("Normalised Importance")
    ax.set_title(f"Top {top_n} Brain Regions — GNN Edge Importance")
    ax.set_xlim(0, top["importance"].max() * 1.15)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Region bar chart → {save_path}")


def plot_importance_std(all_masks, edge_df, top_n=30, save_path=IMPORTANCE_STD_PATH):
    # Bar chart of mean ± 1 SD importance for the top-N edges,
    # showing cross-subject variability in which connections are informative.
    mean_ = all_masks.mean(axis=0)
    std_ = all_masks.std(axis=0)
    top_idx = np.argsort(mean_)[::-1][:top_n]

    labels = []
    for idx in top_idx:
        if idx < len(edge_df):
            row = edge_df.iloc[idx]
            labels.append(
                f"{str(row['source_region'])[:12]}→{str(row['target_region'])[:12]}"
            )
        else:
            labels.append(str(idx))

    x = np.arange(top_n)
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(x, mean_[top_idx], color="steelblue", alpha=0.75,
           width=0.7, label="Mean importance")
    ax.errorbar(x, mean_[top_idx], yerr=std_[top_idx],
                fmt="none", color="black", capsize=3, lw=1.5, label="±1 SD")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=70, ha="right", fontsize=7)
    ax.set_ylabel("Edge Importance")
    ax.set_title(f"Top {top_n} Edges — Mean ± SD across Subjects")
    ax.legend(fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Importance ±SD chart → {save_path}")


# Run Explainer Pipeline

NUM_TOP_EDGES = 100
MAX_SUBJECTS = None  # Set to an integer to explain a subset of subjects
EXPLAIN_STEPS = 400
EXPLAIN_LR = 0.03
SPARSITY_COEF = 0.02

print("Population-level edge importance explanation")

mean_mask, union_src, union_tgt, edge_w_mean, all_masks = explain_population(
    frozen_model = model,
    dataset = val_ds,
    subject_list = list(val_subj),
    scaler = scaler,
    max_subjects = MAX_SUBJECTS,
    steps = EXPLAIN_STEPS,
    lr = EXPLAIN_LR,
    sparsity_coef = SPARSITY_COEF,
)

num_regions = len(region_names)
edge_df, region_df = build_importance_tables(
    mean_mask, union_src, union_tgt, edge_w_mean, region_names
)

print("\nTop 15 most important edges")
print(edge_df[["source_region", "target_region", "weight", "importance"]]
      .head(15).to_string(index=False))

print("\nTop 15 most important regions")
print(region_df.head(15).to_string(index=False))

print("\nTop 15 least important regions")
print(region_df.tail(15).to_string(index=False))

conn_matrix = build_conn_matrix(edge_df, num_regions, num_top=NUM_TOP_EDGES)

print("\Making 3-D connectome")
plot_3d_connectome_inline(
    conn_matrix, coords,
    title=f"GNN Edge Importance — top {NUM_TOP_EDGES} edges",
    edge_thresh="85%", node_size=6, linewidth=3,
)

region_scores_arr = (
    region_df.set_index("region")
             .reindex(region_names)["importance"]
             .fillna(0.0)
             .values
)
plot_2d_glass_brain(conn_matrix, coords, region_scores_arr)
plot_region_bar(region_df, top_n=20)
plot_importance_std(all_masks, edge_df, top_n=30)

print("\nAll outputs written to /content/")

# Save full region ranking to CSV
region_df["rank"] = region_df.index + 1
region_df[["rank", "region", "importance"]].to_csv(REGION_RANKINGS_PATH, index=False)
print(f"Region rankings → {REGION_RANKINGS_PATH}")